# Exploring SU2 CFD meshes with pyramids-mesh

[**SU2**](https://su2code.github.io/) is the open-source suite for computational fluid dynamics (CFD) and
PDE-constrained aerodynamic optimisation. Its native `.su2` file is a plain-text mesh: a list of points, a list of
elements tagged by VTK cell code (triangles = 5, quads = 9, and the 3-D volume codes 10/12/13/14), and named
boundary **markers** (walls, inlets, farfields). Everything is **0-based**.

`pyramids-mesh` is a *file-format reader*: it parses the `.su2` text and hands back a
`pyramids.netcdf.ugrid.MeshDataset` — the same object every other driver produces — so a CFD grid becomes an
ordinary pyramids mesh you can explore, plot and run GIS on. The driver reads **2-D surface meshes only**
(triangles and quads); 3-D volume meshes are declined by design, and the last section shows exactly how.

This notebook works with **real meshes taken verbatim from the official `su2code/TestCases` repository** — the
same grids the SU2 project ships for its tutorials.

## 1. Imports

`MeshDataset` is the object every driver returns. Importing from `pyramids` also puts its vendored GDAL/PROJ data
in place, so nothing here is order-sensitive in a notebook — just import the names you use. `add_tiles` adds a web
basemap *when the mesh is georeferenced*; CFD meshes are not, so we only call it behind an `if ds.epsg` guard.

In [ ]:
from pyramids.netcdf.ugrid import MeshDataset
import numpy as np
import pandas as pd
from cleopatra.basemap.tiles import add_tiles

## 2. Locate the example data

This notebook lives in `docs/examples/real-data/`, so the repo-root `examples/data/real-data/` folder is three
levels up. Paths are written **relative to the notebook itself**, and `read_file` expects a `pathlib.Path`, never
`str(...)`. We start with the NACA 0012 airfoil mesh.

In [ ]:
from pathlib import Path
DATA = Path("../../../examples/data/real-data")
airfoil_file = DATA / "su2" / "naca0012_2d_tri_reads.su2"
airfoil_file, airfoil_file.exists()

## 3. Read the airfoil mesh

`MeshDataset.read_file(path, driver=...)` parses the file and returns a `MeshDataset` — the same type pyramids
produces from a NetCDF UGRID file. We name the `"su2"` driver explicitly and pass the **`Path`** object. This grid
is the classic NACA 0012 inviscid test case: an airfoil in a large circular farfield.

In [ ]:
ds = MeshDataset.read_file(airfoil_file, driver="su2")
type(ds).__name__

Now that a mesh has been read, cleopatra has switched matplotlib to a headless backend. Call its chooser once so
every later figure embeds inline in the notebook (and falls back to `Agg` in a plain script/CI).

In [ ]:
from cleopatra.config import Config
Config.set_matplotlib_backend()

## 4. Explore the dataset

A `MeshDataset` exposes summary properties: `n_node` / `n_face` (mesh size), `crs` / `epsg` (georeferencing if the
format recorded any), `bounds` (the `(minx, miny, maxx, maxy)` extent) and `data_variable_names` (attached field
variables).

In [ ]:
pd.Series({
    "n_node": ds.n_node,
    "n_face": ds.n_face,
    "crs": ds.crs,
    "epsg": ds.epsg,
    "bounds": tuple(round(float(b), 3) for b in ds.bounds),
    "data_variable_names": ds.data_variable_names,
})

5 233 nodes and 10 216 cells — a real working CFD grid. `crs`/`epsg` are `None`: a CFD mesh lives in engineering
(chord-length) coordinates, not a geographic CRS. The extent runs roughly `(-20, -20)` to `(20, 20)` chord
lengths — the airfoil sits at the origin and the farfield boundary is pushed far out, as inviscid external-flow
cases require. `data_variable_names` is empty: the `.su2` file is the grid only; flow fields come from the solver
output, not the mesh.

## 5. Face-degree distribution

`ds.mesh.face_node_connectivity.data` is the `(n_face, k)` table of node indices per face, where `k` is the widest
polygon. Cells with fewer nodes than `k` are **padded with `-1` (fill)**. Counting the non-negative entries per
row gives each face's degree — its number of nodes.

In [ ]:
conn = ds.mesh.face_node_connectivity.data
degree = (conn >= 0).sum(axis=1)
vals, counts = np.unique(degree, return_counts=True)
dist = pd.Series(dict(zip(vals.tolist(), counts.tolist())), name="n_faces")
dist.index.name = "nodes_per_face"
{"degree_distribution": dist.to_dict(), "fill_slots(-1)": int((conn == -1).sum())}

Every one of the 10 216 cells has **3 nodes** with no `-1` fill — a pure **triangular** mesh (VTK code 5). We will
meet an all-quad SU2 mesh (degree 4) further down, which is why this count matters: it is what decides whether
certain GIS operations apply.

## 6. Wireframe: `plot_outline()`

The simplest view needs no data — just the mesh edges. `plot_outline()` draws the wireframe through cleopatra and
embeds it inline. For an airfoil grid this is the fastest way to *see* the aerodynamics: triangles cluster tightly
around the profile where gradients are steep and coarsen outward toward the farfield.

In [ ]:
_g = ds.plot_outline()
if ds.epsg:
    add_tiles(_g.ax, crs=ds.epsg)

The dense dark knot at the centre is the airfoil and its boundary-layer refinement; the mesh fans out to the
sparse farfield ring. This single read-and-draw is the whole point of the driver: a CFD grid rendered with the
same `MeshDataset` call as any other format.

The farfield is so large it hides the profile. Clip to a unit window around the origin with `subset_by_bounds`
(four positional coordinates) and redraw — now the airfoil shape itself is visible.

In [ ]:
near = ds.subset_by_bounds(-0.6, -0.6, 1.6, 0.6)
_g = near.plot_outline()
if ds.epsg:
    add_tiles(_g.ax, crs=ds.epsg)
{"full_n_face": ds.n_face, "near_airfoil_n_face": near.n_face}

Clipped to the chord, the NACA 0012 outline emerges — the characteristic symmetric airfoil wrapped in its
O-grid of triangles. `subset_by_bounds` returns a new `MeshDataset`, so the clip is itself a first-class mesh.

## 7. A data map: synthesize and colour a per-cell field

To draw a *filled* map we need a value per cell. The mesh carries geometry only, so we compute one — each cell's
**area** — and attach it with `with_variable(..., location="face")`, which returns a **new** dataset (the original
`ds` is untouched). Then `plot` colours the mesh by it.

In [ ]:
d2 = ds.with_variable("cell_area", ds.mesh.face_areas, location="face")
_g = d2.plot("cell_area", cmap="viridis", colorbar=True, title="Per-cell area (NACA 0012)")
if ds.epsg:
    add_tiles(_g.ax, crs=ds.epsg)

The map is almost uniform because the cell areas span many orders of magnitude — tiny near the airfoil, huge at
the farfield — so a linear colour scale is dominated by the few large outer cells. That dynamic range is itself
the physics: the solver needs fine cells where the flow changes fastest. This is the generic pyramids data-map
path — any per-cell field (pressure, Mach number, water depth) plots exactly the same way.

With a field attached, `.summary()` reports per-variable statistics. It confirms the range numerically — the
smallest cells are ~`1e-7`, the largest ~`4`, a seven-order-of-magnitude spread.

In [ ]:
d2.summary()

## 8. GIS operations, for free

Because the mesh is now a first-class pyramids object, the pyramids GIS layer applies directly — `pyramids-mesh`
itself adds none of this. Each cell below runs one operation and interprets it.

### Faces as a GeoDataFrame

`to_geodataframe()` turns every face into a polygon row of a geopandas `GeoDataFrame` — the standard vector
structure for export to shapefile/GeoPackage or for spatial joins. One row per triangle.

In [ ]:
gdf = ds.to_geodataframe()
{"rows": len(gdf), "columns": list(gdf.columns)}

10 216 polygon rows — one per triangular cell — ready for the shapely/geopandas ecosystem.

### Bounding polygon

`bounding_polygon()` dissolves the cells into a single shapely polygon of the meshed footprint — the gateway to
clipping and spatial joins. For this case it is the circular farfield disc.

In [ ]:
poly = ds.bounding_polygon()
poly.geom_type, tuple(round(float(b), 3) for b in poly.bounds)

### Circumcenters — defined here

`circumcenters()` returns one circumcentre per cell. It is **only defined for triangular meshes** (a triangle has
a unique circumscribed circle; a general quad does not). This mesh is all triangles, so the call succeeds — one
`(x, y)` per face. These points are what a finite-volume scheme uses to build orthogonal dual cells.

In [ ]:
cc = ds.circumcenters()
{"shape": cc.shape, "first_three": np.round(cc[:3], 4).tolist()}

One circumcentre per triangle, `(10216, 2)`. Hold on to the fact that this *worked* — the next section reads an
all-quad SU2 mesh where the very same call is refused.

## 9. A different topology: the quad channel mesh

SU2 meshes are not all triangles. `channel_2d_quad_reads.su2` is a structured **quadrilateral** grid of a plane
channel (markers `inlet`, `outlet`, `lower_wall`, `upper_wall`). Reading it uses the identical call — only the
cell shape differs.

In [ ]:
channel_file = DATA / "su2" / "channel_2d_quad_reads.su2"
cds = MeshDataset.read_file(channel_file, driver="su2")
cconn = cds.mesh.face_node_connectivity.data
cdeg = (cconn >= 0).sum(axis=1)
cvals, ccounts = np.unique(cdeg, return_counts=True)
pd.Series({
    "n_node": cds.n_node,
    "n_face": cds.n_face,
    "epsg": cds.epsg,
    "bounds": tuple(round(float(b), 3) for b in cds.bounds),
    "degree_distribution": dict(zip(cvals.tolist(), ccounts.tolist())),
})

32 768 nodes, 32 385 cells, every one **degree 4** — a pure quad grid on the `3 x 1` channel. Same `MeshDataset`
API, different element type.

A cell-area map shows the channel's structure directly: the grid is refined toward the walls (thin cells) and
coarser in the core, the standard way to resolve a wall boundary layer.

In [ ]:
c2 = cds.with_variable("cell_area", cds.mesh.face_areas, location="face")
_g = c2.plot("cell_area", cmap="viridis", colorbar=True, title="Per-cell area (channel)")
if cds.epsg:
    add_tiles(_g.ax, crs=cds.epsg)

The wide `3:1` channel shows horizontal bands of near-constant area, darkest (smallest cells) along the top and
bottom walls — the refined boundary-layer rows — and lighter through the middle.

### Circumcenters — refused for quads

Now the contrast promised above. `circumcenters()` has no meaning for a quadrilateral, so on this mesh it raises
`ValueError`. We catch it rather than crash — for a quad grid use the cell **centroids** (`mesh.face_centroids`)
instead.

In [ ]:
try:
    cds.circumcenters()
except ValueError as e:
    print("quad mesh -> circumcenters skipped:", e)

cx, cy = cds.mesh.face_centroids
print("centroids available instead, shape:", cx.shape, cy.shape)

Same operation, opposite outcome: it succeeded on the triangle mesh and is declined on the quad one. The topology
the file encodes — not the driver — decides which geometric constructions are defined.

## 10. The limit: 3-D volume meshes are declined

SU2 also writes **3-D volume meshes** (`NDIME=3`): tetrahedra (VTK 10), hexahedra (12), prisms (13), pyramids
(14). `pyramids-mesh` is a reader for **2-D surface meshes**, so it declines every 3-D volume file with an
explicit error rather than silently dropping the third dimension. We catch `FileFormatNotSupportedError` (the base
class these errors derive from), loop over three real rejected files, and tabulate what each one reported.

In [ ]:
import re
from pyramids.base._errors import FileFormatNotSupportedError

rejected = [
    "brick_3d_tet_rejected.su2",
    "brick_3d_hex_rejected.su2",
    "synthetic_3d_pyra-only_rejected.su2",
]
rows = []
for name in rejected:
    try:
        MeshDataset.read_file(DATA / "su2" / name, driver="su2")
        rows.append({"file": name, "raised": "(none - read!)", "cell_type": "", "vtk_code": ""})
    except FileFormatNotSupportedError as e:
        m = re.search(r"holds (\w+) cells \(VTK code (\d+)\)", str(e))
        rows.append({
            "file": name,
            "raised": type(e).__name__,
            "cell_type": m.group(1) if m else "?",
            "vtk_code": m.group(2) if m else "?",
        })
pd.DataFrame(rows).set_index("file")

All three are refused with the same error — concretely an `UnsupportedMeshFormatError`, a subclass of the
`FileFormatNotSupportedError` we caught — each naming the volume cell type it found: tetrahedron (10), hexahedron
(12), pyramid (14). The rejection is driven by the element codes in the file, not by the filename.

The full message is deliberately explicit — here is one in full, so you can see exactly what the reader tells a
user who hands it a volume mesh.

In [ ]:
try:
    MeshDataset.read_file(DATA / "su2" / "brick_3d_tet_rejected.su2", driver="su2")
except FileFormatNotSupportedError as e:
    msg = str(e)
    # show the explanatory sentence, from "is a 3-D volume mesh" onward
    print(msg[msg.index("is a 3-D volume mesh"):].rstrip('")'))

Why decline instead of read? A 3-D volume mesh has no single 2-D surface to represent as a `Mesh2d` — its cells
fill a volume. Flattening it would silently discard real topology, so the driver stops with a clear message.
Extracting the boundary *surface* of a volume mesh is a mesh operation that belongs upstream in `pyramids-gis`,
not in a format reader.

## Takeaway

- **SU2 `.su2` = native CFD mesh**: VTK-coded, 0-based points and elements, read into a `MeshDataset` with
  `MeshDataset.read_file(path, driver="su2")`.
- **Real 2-D meshes read cleanly**: the NACA 0012 airfoil (5 233 nodes, 10 216 **triangles**) and a plane
  channel (32 768 nodes, 32 385 **quads**) — explore, plot and GIS all work on the CFD grid as on any mesh.
- **Topology decides the geometry**: `circumcenters()` is defined for the triangle mesh but refused for the quad
  mesh; use `mesh.face_centroids` there instead.
- **No CRS**: CFD meshes carry engineering coordinates (`epsg`/`crs` are `None`), so reprojection and basemap
  tiles are skipped.
- **3-D volume meshes are declined by design**: every `NDIME=3` file (tet/hex/prism/pyramid) raises an
  `UnsupportedMeshFormatError` (a `FileFormatNotSupportedError`) naming the offending VTK code — `pyramids-mesh`
  reads 2-D surface meshes only.